# 5.0 — LocalLLMNormalizer Inference Test (llama.cpp)

End-to-end exercise of the term normalizer: SapBERT candidate retrieval plus
a local LLM via `llama.cpp` (GGUF, Gemma 3n E4B-IT, Q8) that selects the
index of the best candidate.

If the `.gguf` referenced by `LLM_MODEL_PATH` is not present on disk, the
inference cells are **skipped** with a message and the notebook completes
without error.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent  # notebooks/ -> repo root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import time

## Closed-Vocabulary Test Pool (MedDRA)

A hand-curated pool of MedDRA Preferred Terms with illustrative codes (in
production this vocabulary comes from SIDER via `build_meddra_vocab`). Each
run draws a random subset of 15-20 terms from the pool, so the candidate list
the LLM ranks against varies across executions -- useful for spot-checking
ranking behavior against different vocabularies without depending on the full
dataset.

In [5]:
import random

# Illustrative pool; real codes/PTs come from SIDER at ETL time.
MEDDRA_POOL: dict[str, str] = {
    "Insomnia": "10022437",
    "Headache": "10019211",
    "Nausea": "10028813",
    "Gastrointestinal haemorrhage": "10017955",
    "Diarrhoea": "10012735",
    "Rash": "10037844",
    "Fatigue": "10016256",
    "Dizziness": "10013573",
    "Vomiting": "10047700",
    "Pruritus": "10037087",
    "Abdominal pain": "10000081",
    "Anxiety": "10002855",
    "Constipation": "10010774",
    "Cough": "10011224",
    "Dyspnoea": "10013968",
    "Hypotension": "10021097",
    "Hypertension": "10020772",
    "Tachycardia": "10043071",
    "Bradycardia": "10006093",
    "Anaemia": "10002034",
    "Thrombocytopenia": "10043554",
    "Neutropenia": "10029354",
    "Hepatotoxicity": "10019851",
    "Renal impairment": "10062237",
    "Peripheral oedema": "10034972",
    "Somnolence": "10041349",
    "Tremor": "10044565",
    "Myalgia": "10028411",
    "Arthralgia": "10003239",
    "Pyrexia": "10037660",
    "Chills": "10008531",
    "Weight decreased": "10047895",
    "Weight increased": "10047899",
    "Blurred vision": "10047513",
    "Tinnitus": "10043882",
    "Dry mouth": "10013781",
    "Urticaria": "10046735",
    "Palpitations": "10033557",
    "Syncope": "10042772",
    "Confusional state": "10010305",
}

SEED: int | None = None  # set an int for a reproducible sample across reruns

_rng = random.Random(SEED)
SAMPLE_SIZE = _rng.randint(15, 20)
sampled_terms = _rng.sample(list(MEDDRA_POOL), k=min(SAMPLE_SIZE, len(MEDDRA_POOL)))
MEDDRA_VOCAB = {term: MEDDRA_POOL[term] for term in sampled_terms}

print(f"Sampled {len(MEDDRA_VOCAB)} / {len(MEDDRA_POOL)} terms (seed={SEED}):")
for term, code in MEDDRA_VOCAB.items():
    print(f"  {code:>10}  {term}")

10

## Stage 1 — Retrieval (SapBERT)

Loads the biomedical bi-encoder and exercises `candidates()` on colloquial
phrases that share no literal text with the target Preferred Term -- the
real-world case being openFDA label wording.

In [3]:
generator = SapBERTCandidateGenerator(MEDDRA_VOCAB)

TEST_PHRASES = [
    "can't sleep",
    "throwing up",
    "stomach bleeding",
    "skin itching all over",
    "feeling extremely tired",
]

for phrase in TEST_PHRASES:
    top = generator.candidates(phrase, k=3)
    print(f"{phrase!r:35} -> {top}")

2026-07-21 07:10:47,894 | INFO     | sentence_transformers.base.model | No device provided, using cuda:0
2026-07-21 07:10:48,111 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-from-PubMedBERT-fulltext/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-07-21 07:10:48,115 | INFO     | sentence_transformers.base.model | No modules.json found for cambridgeltl/SapBERT-from-PubMedBERT-fulltext, initializing a new SentenceTransformer model.
2026-07-21 07:10:48,259 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-from-PubMedBERT-fulltext/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-07-21 07:10:48,405 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-from-PubMedBERT-fulltext/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-21 07:10:48,559 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cambridgeltl/SapBERT-from

config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

2026-07-21 07:10:48,909 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-from-PubMedBERT-fulltext/resolve/main/model.safetensors "HTTP/1.1 302 Found"


model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

...

'stomach bleeding'                  -> [('Gastrointestinal haemorrhage', '10017955'), ('Vomiting', '10047700'), ('Nausea', '10028813')]


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

'skin itching all over'             -> [('Pruritus', '10037087'), ('Rash', '10037844'), ('Dizziness', '10013573')]


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

'feeling extremely tired'           -> [('Fatigue', '10016256'), ('Nausea', '10028813'), ('Dizziness', '10013573')]


## Stage 2 — Ranking (Local LLM, llama.cpp)

Instantiates `LocalLLMNormalizer` over the `.gguf` configured in
`LLM_MODEL_PATH`. Loads the model into memory once; this cell may take a
while.

In [6]:
NORMALIZER: LocalLLMNormalizer | None = None
MODEL_PATH = Path(LLM_MODEL_PATH)

if not MODEL_PATH.exists():
    print(f"Model skipped — {MODEL_PATH} not found.")
else:
    t0 = time.perf_counter()
    NORMALIZER = LocalLLMNormalizer(generator)
    print(f"Model loaded in {time.perf_counter() - t0:.2f}s")

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


Model loaded in 2.21s


## End-to-End Inference

`normalize()` chains retrieval, prompt construction, grammar-constrained
inference (`[0-9]+`), and index parsing. Prints each phrase, the chosen
term/code, and per-call latency.

In [7]:
if NORMALIZER is None:
    print("Model skipped — no inference to run.")
else:
    for phrase in TEST_PHRASES:
        t0 = time.perf_counter()
        result = NORMALIZER.normalize(phrase)
        elapsed = time.perf_counter() - t0
        print(f"{phrase!r:35} -> {result}  ({elapsed:.2f}s)")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

"can't sleep"                       -> ('Insomnia', '10022437')  (1.28s)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

'throwing up'                       -> ('Vomiting', '10047700')  (0.59s)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

'stomach bleeding'                  -> ('Gastrointestinal haemorrhage', '10017955')  (0.61s)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

'skin itching all over'             -> ('Pruritus', '10037087')  (0.60s)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

'feeling extremely tired'           -> ('Fatigue', '10016256')  (0.57s)


## Inspecting the Raw Response

Repeats a single call while exposing the prompt sent to the model and its
raw response (before `_parse_index` runs), confirming that the grammar
constrains generation to digits only.

In [ ]:
if NORMALIZER is None:
    print("Model skipped — no raw response to show.")
else:
    phrase = TEST_PHRASES[0]
    candidates = NORMALIZER.generator.candidates(phrase, NORMALIZER.top_k)
    prompt = NORMALIZER._build_prompt(phrase, candidates)
    print("--- PROMPT ---")
    print(prompt)
    print("="*50)

    completion = NORMALIZER.llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        temperature=NORMALIZER.params["temperature"],
        top_k=NORMALIZER.params["top_k"],
        top_p=NORMALIZER.params["top_p"],
        min_p=NORMALIZER.params["min_p"],
        repeat_penalty=NORMALIZER.params["repeat_penalty"],
        max_tokens=4,
        grammar=NORMALIZER.grammar,
    )
    raw = completion["choices"][0]["message"]["content"]
    print("--- RAW ANSWER ---")
    print(repr(raw))
    print("--- PARSED INDEX ---")
    print(NORMALIZER._parse_index(raw))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

--- PROMPT ---
You are a strict medical coding assistant performing entity linking between free-text adverse-reaction descriptions and MedDRA Preferred Terms (PTs).

Rules:
- Choose the single candidate PT that best matches the MEDICAL MEANING of the adverse-reaction text, not surface wording similarity.
- A match must be a legitimate synonym, abbreviation, lay term, or narrower/broader clinical expression of the SAME clinical concept. Do not match based on partial word overlap alone.
- If two candidates seem plausible, choose the more clinically precise one, not the most general.
- If the text describes a concept not represented by any candidate, or is too vague/ambiguous to map confidently to one specific PT, answer 0. It is always better to answer 0 than to guess.
- Do not use outside medical knowledge to reinterpret the text beyond what it states. Do not infer a diagnosis or mechanism not implied by the wording itself.

Output format:
- Answer with ONLY the number of the best-match

## Edge Case

A phrase with no plausible match in the closed vocabulary should return
`None` (candidate 0 / NONE) rather than forcing a choice.

In [10]:
if NORMALIZER is None:
    print("Model skipped — no edge case to run.")
else:
    edge_phrase = "increased appetite for spicy food"
    print(NORMALIZER.normalize(edge_phrase))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

None
